In [1]:
import numpy as np
import pandas as pd

In [2]:
users = pd.read_csv("../data/processed/users_cleaned.csv")
orders = pd.read_csv("../data/processed/orders_cleaned.csv")


In [4]:
print("Users:", users.shape)
print("Orders:", orders.shape)

Users: (100000, 6)
Orders: (148528, 6)


ORDERS

In [5]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

CALCULATING TOTAL ORDERS

In [6]:
customer_features = (orders.groupby("user_id").agg(total_orders=("user_id", "count")).reset_index())

customer_features.head()

,user_id,total_orders
0,1,2
1,2,3
2,3,1
3,4,1
4,5,2


In [7]:
customer_features.shape

(77546, 2)

In [8]:
customer_features["total_orders"].describe()

count    77546.000000
mean         1.915353
std          1.034590
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          8.000000
Name: total_orders, dtype: float64

CALCULATING TOTAL SPENDING

In [9]:
spending = (orders.groupby("user_id").agg(total_spend=("sales_amount", "sum")).reset_index())

customer_features = customer_features.merge(spending,on="user_id",how="left")

customer_features.head()

,user_id,total_orders,total_spend
0,1,2,1672
1,2,3,4061
2,3,1,21542
3,4,1,102
4,5,2,52047


CALCULATING TOTAL QUANTITY

In [10]:
quantity = (orders.groupby("user_id").agg(total_quantity = ("sales_qty","sum")).reset_index())

customer_features = customer_features.merge(quantity,on = "user_id",how = "left")
customer_features.head()

,user_id,total_orders,total_spend,total_quantity
0,1,2,1672,5
1,2,3,4061,7
2,3,1,21542,500
3,4,1,102,1
4,5,2,52047,50


AVERAGE ORDER VALUE

In [11]:
customer_features["average_order_value"] = (customer_features["total_spend"] / customer_features["total_orders"])

In [12]:
customer_features.head()

,user_id,total_orders,total_spend,total_quantity,average_order_value
0,1,2,1672,5,836.000000
1,2,3,4061,7,1353.666667
2,3,1,21542,500,21542.000000
3,4,1,102,1,102.000000
4,5,2,52047,50,26023.500000


FIRST ORDER DATE

In [13]:
first_order = (orders.groupby("user_id")["order_date"].min().reset_index()
    .rename(columns={"order_date": "first_order_date"}))

customer_features = customer_features.merge(first_order,on="user_id",how="left")

LAST ORDER DATE

In [14]:
last_order = (orders.groupby("user_id")["order_date"].max().reset_index()
    .rename(columns={"order_date": "last_order_date"}))

customer_features = customer_features.merge(last_order,on="user_id",how="left")

In [15]:
customer_features.head()

,user_id,total_orders,total_spend,total_quantity,average_order_value,first_order_date,last_order_date
0,1,2,1672,5,836.000000,2017-12-12,2018-11-14
1,2,3,4061,7,1353.666667,2018-02-13,2018-08-03
2,3,1,21542,500,21542.000000,2019-05-15,2019-05-15
3,4,1,102,1,102.000000,2019-02-06,2019-02-06
4,5,2,52047,50,26023.500000,2018-03-02,2018-07-25


In [16]:
customer_features = customer_features.merge(users[["user_id", "age", "gender", "marital_status", "occupation"]],
                    on="user_id",how="left")

In [17]:
customer_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77546 entries, 0 to 77545
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              77546 non-null  int64         
 1   total_orders         77546 non-null  int64         
 2   total_spend          77546 non-null  int64         
 3   total_quantity       77546 non-null  int64         
 4   average_order_value  77546 non-null  float64       
 5   first_order_date     77546 non-null  datetime64[ns]
 6   last_order_date      77546 non-null  datetime64[ns]
 7   age                  77546 non-null  int64         
 8   gender               77546 non-null  object        
 9   marital_status       77546 non-null  object        
 10  occupation           77546 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(5), object(3)
memory usage: 6.5+ MB


In [18]:
customer_features.isnull().sum()

user_id                0
total_orders           0
total_spend            0
total_quantity         0
average_order_value    0
first_order_date       0
last_order_date        0
age                    0
gender                 0
marital_status         0
occupation             0
dtype: int64

In [19]:
customer_features["user_id"].duplicated().sum()

np.int64(0)

In [20]:
customer_features.columns

Index(['user_id', 'total_orders', 'total_spend', 'total_quantity',
       'average_order_value', 'first_order_date', 'last_order_date', 'age',
       'gender', 'marital_status', 'occupation'],
      dtype='object')

In [21]:
customer_features.to_csv("../data/processed/customer_features.csv",index=False)

In [22]:
customer_features.shape

(77546, 11)